![Logo 1](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech1.jpg)
<div class="alert alert-block alert-info">
<h1> Komputerowe wspomaganie tłumaczenia </h1>
<h2> 8. <i>Wykorzystanie tłumaczenia automatycznego we wspomaganiu tłumaczenia</i> [laboratoria]</h2> 
<h3>Rafał Jaworski (2021)</h3>
</div>

![Logo 2](https://git.wmi.amu.edu.pl/AITech/Szablon/raw/branch/master/Logotyp_AITech2.jpg)

W dzisiejszych czasach, niezwykle ważną techniką wspomagania tłumaczenia jest użycie tłumaczenia maszynowego. Tekst źródłowy do tłumaczenia jest najpierw tłumaczony całkowicie autommatycznie, a następnie tłumacz ludzki dokonuje korekty wyniku. Technologia tłumaczenia maszynowego jest już na tyle dojrzała, że oferuje bardzo wysoką jakość wyników. Coraz częstsze stają się scenariusze, w których ludzka korekta to niemal całkowicie machinalne (sic!) zatwierdzanie wyników tłumaczenia maszynowego. Na dzisiejszych zajęciach poznamy techniki ewaluacji tłumaczenia maszynowego oraz sprawdzania jego przydatności w procesie wspomagania tłumaczenia ludzkiego.

Jakość tłumaczenia maszynowego możemy oceniać na dwóch niezależnych płaszczyznach: dokładność i płynność. Płynność jest subiektywnie odbieranym odczuciem, że czytany tekst jest napisany językiem naturalnym i zrozumiałym. Systemy tłumaczenia maszynowego oparte na uczeniu głębokim z wykorzystaniem sieci neuronowych osiągają duży stopień płynności tłumaczenia. Niestety jednak ich dokładność nie zawsze jest równie wysoka.

Dokładność tłumaczenia maszynowego jest parametrem, który łatwiej zmierzyć. Wartość takich pomiarów daje obraz tego, jaka jest faktyczna jakość tłumaczenia maszynowego i jaka jest jego potencjalna przydatność we wspomaganiu tłumaczenia.

Najczęściej stosowaną techniką oceny tłumaczenia maszynowego jest ocena BLEU. Do obliczenia tej oceny potrzebny jest wynik tłumaczenia maszynowego oraz referencyjne tłumaczenie ludzkie wysokiej jakości.

### Ćwiczenie 1: Zaimplementuj program do obliczania oceny BLEU dla korpusu w folderze data. Użyj implementacji BLEU z pakietu nltk. Dodatkowe wymaganie techniczne - napisz program tak, aby nie musiał rozpakwowywać pliku zip z korpusem na dysku.

In [8]:
import zipfile
import os
from nltk.translate.bleu_score import corpus_bleu

def calculate_bleu(zip_path: str):
    if not os.path.exists(zip_path):
        return f"Zip not found: {zip_path}"

    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            all_files = z.namelist()

            hyp_file = next((f for f in all_files if f.endswith('_nmt.txt')), None)
            ref_file = next((f for f in all_files if f.endswith('_human.txt')), None)

            if not hyp_file or not ref_file:
                return f"Files not found, files in ziP: {all_files}"


            with z.open(hyp_file) as f_hyp, z.open(ref_file) as f_ref:
                hyp_lines = f_hyp.read().decode('utf-8').splitlines()
                ref_lines = f_ref.read().decode('utf-8').splitlines()

        hypotheses = [line.split() for line in hyp_lines]
        references = [[line.split()] for line in ref_lines]

        return corpus_bleu(references, hypotheses)

    except Exception as e:
        return e

sciezka_do_zip = "/home/dbanaszak/projects/ai_translation_support/aitech-kwt/lab/data/corpus_corrected.zip"
result = calculate_bleu(sciezka_do_zip)

print(f"BLEU: {result}")

BLEU: 0.7675889499772856


### Ćwiczenie 2: Oblicz wartość bleu na różnych fragmentach przykładowego korpusu (np. na pierwszych 100 zdaniach, zdaniach 500-600). Czy w jakimś fragmencie korpusu jakość tłumaczenia znacząco odbiega od średniej?

In [11]:
import zipfile
import os
from nltk.translate.bleu_score import corpus_bleu

def load_corpus_from_zip(zip_path):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Archive not found: {zip_path}")

    with zipfile.ZipFile(zip_path, 'r') as z:
        filenames = z.namelist()
        hyp_path = next((f for f in filenames if f.endswith('_nmt.txt')), None)
        ref_path = next((f for f in filenames if f.endswith('_human.txt')), None)

        if not hyp_path or not ref_path:
            raise ValueError("Required files (_nmt.txt and _human.txt) missing in ZIP.")

        with z.open(hyp_path) as f_hyp, z.open(ref_path) as f_ref:
            hyp_lines = f_hyp.read().decode('utf-8').splitlines()
            ref_lines = f_ref.read().decode('utf-8').splitlines()

    return hyp_lines, ref_lines

def get_bleu_score(hypotheses, references, start=0, end=None):
    if end is None:
        end = len(hypotheses)

    hyp_slice = [line.split() for line in hypotheses[start:end]]
    ref_slice = [[line.split()] for line in references[start:end]]

    if not hyp_slice:
        return 0.0

    return corpus_bleu(ref_slice, hyp_slice)

# Execution
zip_file_path = "../lab/data/corpus_corrected.zip"
hyp_all, ref_all = load_corpus_from_zip(zip_file_path)

total_score = get_bleu_score(hyp_all, ref_all)
test_ranges = [(0, 100), (100, 200), (200, 300), (500, 600), (len(hyp_all)-100, len(hyp_all))]

print(f"Total BLEU Score: {total_score:.4f}\n")
print(f"{'Sentence Range':<20} | {'BLEU':<10} | {'Deviation'}")
print("-" * 45)

for start, end in test_ranges:
    score = get_bleu_score(hyp_all, ref_all, start, end)
    deviation = score - total_score
    print(f"{start:>4} - {end:>4}        | {score:.4f}     | {deviation:+.4f}")

Total BLEU Score: 0.7676

Sentence Range       | BLEU       | Deviation
---------------------------------------------
   0 -  100        | 0.7763     | +0.0087
 100 -  200        | 0.7835     | +0.0159
 200 -  300        | 0.7444     | -0.0232
 500 -  600        | 0.7969     | +0.0293
 900 - 1000        | 0.7611     | -0.0065


Inną metodą oceny jakości tłumaczenia maszynowego jest parametr WER - Word Error Rate. Definiuje się on w następujący sposób:

$WER = \frac{S+D+I}{N}=\frac{S+D+I}{S+D+C}$

gdzie:
 * S - liczba substytucji (słów)
 * D - liczba usunięć
 * I - liczba wstawień
 * C - liczba poprawnych śłów
 * N - liczba słów w tłumaczeniu referencyjnym (N=S+D+C)

Miara ta jest zwykle używana w do oceny systemów automatycznego rozpoznawania mowy, jednak w kontekście wspomagania tłumaczenia może być rozumiana jako wielkość nakładu pracy tłumacza nad poprawieniem tłumaczenia maszynowego.

### Ćwiczenie 3: Oblicz wartość WER dla przykładowego korpusu. Skorzystaj z gotowej implementacji WER.

In [15]:
from jiwer import wer


def calculate_wer(hypotheses, references):
    if len(hypotheses) != len(references):
        raise ValueError("Hypotheses and references must have the same number of lines.")

    return wer(references, hypotheses)

# Execution
zip_file_path = "../lab/data/corpus_corrected.zip"
hyp_all, ref_all = load_corpus_from_zip(zip_file_path)

total_wer = calculate_wer(hyp_all, ref_all)

print(f"Word Error Rate (WER) for the corpus: {total_wer:.4f}")
print(f"Percentage: {total_wer * 100:.2f}%")

Word Error Rate (WER) for the corpus: 0.1597
Percentage: 15.97%


Poza wymienionymi powyżej, stosować można jeszcze inne miary oparte na porównywaniu tłumaczenia maszynowego z ludzkim. Przypomnijmy sobie jedną, którą stosowaliśmy wcześniej.

### Ćwiczenie 4: Oblicz średnią wartość dystansu Levenshteina pomiędzy zdaniami przetłumaczonymi automatycznie oraz przez człowieka. Użyj implementacji z ćwiczeń nr 2.

In [14]:
import Levenshtein

def calculate_average_levenshtein(hypotheses, references):
    if len(hypotheses) != len(references):
        raise ValueError("Dataset size mismatch.")

    total_distance = 0
    for hyp, ref in zip(hypotheses, references):
        total_distance += Levenshtein.distance(hyp, ref)

    return total_distance / len(hypotheses)

# Execution
zip_file_path = "../lab/data/corpus_corrected.zip"
hyp_all, ref_all = load_corpus_from_zip(zip_file_path)

avg_dist = calculate_average_levenshtein(hyp_all, ref_all)

print(f"Average Levenshtein Distance: {avg_dist:.2f}")

Average Levenshtein Distance: 14.69


A teraz sprawdźmy coś jeszcze. W danych przykładowego korpusu znajduje się także angielski tekst źródłowy. Teoretycznie, dobre tłumaczenie niemieckie powinno zawierać jak najwięcej słów z angielskiego źródła. Wykonajmy najstępujący eksperyment:

### Ćwiczenie 5: Dla każdej trójki zdań z korpusu przykładowego wykonaj następujące kroki:
 * Przetłumacz każde angielskie słowo na niemiecki przy użyciu modułu PyDictionary.
 * Sprawdź, które z niemieckich tłumaczeń zawiera więcej spośród tych przetłumaczonych słów - automatyczne, czy ludzkie.
Następnie wypisz statystyki zbiorcze. Które tłumaczenie zawiera więcej słownikowych tłumaczeń słów ze źródła?

In [29]:
import zipfile
import os
from PyDictionary import PyDictionary

def load_full_corpus(zip_path):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Archive not found: {zip_path}")

    with zipfile.ZipFile(zip_path, 'r') as z:
        filenames = z.namelist()

        en_path = next((f for f in filenames if f.endswith('corpus_en.txt')), None)
        hyp_path = next((f for f in filenames if f.endswith('_nmt.txt')), None)
        ref_path = next((f for f in filenames if f.endswith('_human.txt')), None)

        def read_lines(path):
            if path is None:
                return []
            with z.open(path) as f:
                return f.read().decode('utf-8').splitlines()

        return read_lines(en_path), read_lines(hyp_path), read_lines(ref_path)
import re

def analyze_dictionary_hits(en_lines, hyp_lines, ref_lines, limit=3):
    dictionary = PyDictionary()
    stats = {"nmt_wins": 0, "human_wins": 0, "draws": 0}

    for i in range(min(len(en_lines), limit)):
        en_words = re.findall(r'\b\w+\b', en_lines[i].lower())
        hyp_words = set(re.findall(r'\b\w+\b', hyp_lines[i].lower()))
        ref_words = set(re.findall(r'\b\w+\b', ref_lines[i].lower()))

        possible_translations = set()
        for word in en_words:
            try:
                print(word)
                trans = dictionary.translate(word, 'de')
                if trans:
                    for t in trans:
                        clean_t = "".join(filter(str.isalpha, t)).lower()
                        if clean_t:
                            possible_translations.add(clean_t)
            except:
                continue

        nmt_hits = sum(1 for w in hyp_words if w in possible_translations)
        ref_hits = sum(1 for w in ref_words if w in possible_translations)

        print(f"Sentence {i+1}: NMT hits: {nmt_hits}, Human hits: {ref_hits}")

        if nmt_hits > ref_hits:
            stats["nmt_wins"] += 1
        elif ref_hits > nmt_hits:
            stats["human_wins"] += 1
        else:
            stats["draws"] += 1

    return stats

In [28]:
zip_file_path = "../lab/data/corpus_corrected.zip"
try:
    en_all, hyp_all, ref_all = load_full_corpus(zip_file_path)
    results = analyze_dictionary_hits(en_all, hyp_all, ref_all, limit=3)
    print(f"Results: {results}")
except Exception as e:
    print(f"Error: {e}")

press
Invalid Word
h
Invalid Word
while
Invalid Word
in
Invalid Word
review
Invalid Word
mode
Invalid Word
to
Invalid Word
display
Invalid Word
keyboard
Invalid Word
shortcuts
Invalid Word
for
Invalid Word
working
Invalid Word
in
Invalid Word
review
Invalid Word
mode
Invalid Word
Sentence 1: NMT hits: 0, Human hits: 0
click
Invalid Word
remove
Invalid Word
shortcut
Invalid Word
Sentence 2: NMT hits: 0, Human hits: 0
selects
Invalid Word
the
Invalid Word
non
Invalid Word
selected
Invalid Word
areas
Invalid Word
Sentence 3: NMT hits: 0, Human hits: 0
Results: {'nmt_wins': 0, 'human_wins': 0, 'draws': 3}
